In [ ]:
#%matplotlib inline

import os
from collections import Counter
from itertools import count
from random import randint
from pathlib import Path
import math

# Only needed to embed the images in notebook so they
# show up in gitlab repo
from IPython.display import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde
import seaborn

from srim import TRIM, Target, Layer, Ion, Element


In [ ]:

# directory to srim executable (must be SRIM 2013)
SRIM_EXECUTABLE_DIRECTORY = os.path.abspath(r'C:\Program Files\SRIM-2013')

# directory to store srim output (does not have to exist)
#DATA_DIRECTORY = os.path.abspath(r'c:\Users\Sacha\Desktop\nazar\examples\outputs\')
DATA_DIRECTORY = Path("../../../Desktop/nazar/examples/outputs").resolve()
# directory to store images from processing srim output (does not have to exist)
IMAGE_DIRECTORY = os.path.abspath(r'C:\Users\Sacha\Desktop\nazar\examples\images')

os.makedirs(DATA_DIRECTORY, exist_ok=True)
os.makedirs(IMAGE_DIRECTORY, exist_ok=True)

In [ ]:
# Define calculations in code
ions = [
    {'identifier': 'Au', 'energy': 25.0e6}, # eV
    {'identifier': 'C', 'energy': 4.5e6}, # eV
    {'identifier': 'O', 'energy': 6.5e6}, # eV
    {'identifier': 'Si', 'energy': 21.0e6}, # eV
    {'identifier': 'Ni', 'energy': 21.0e6}, # eV
]

layer = Layer({
    'Si': {
        'stoich': 0.5,
        'E_d': 35.0, # Displacement Energy
        'lattice': 0.0,
        'surface': 3.0
    },
    'C': {
        'stoich': 0.5,
        'E_d': 20.0, # Displacement Energy
        'lattice': 0.0,
        'surface': 3.0
    }
}, density=3.21, width=10000.0) #width in Ängstrom /!\
target = Target([layer])

# number of ions to use in each SRIM calculation
# allows for fragmentation of calcluation
# step size should never be larger than 1,000
# if you are storing COLLISON.txt
IONS_PER_STEP = 20
# number of ions to simulate for each ion
NUM_IONS = 100

# See TRIMSettings and TRIM for full options
# you can do any setting that you could do in TRIM
TRIM_SETTINGS = {
    'calculation': 2, # full cascade
    'collisions': 2,
    # 'angle_ions': 30
}

In [ ]:
# Helper functions to run calculations
def fragment(step, total):
    remaining = total
    while remaining > 0:
        if step > remaining:
            return remaining
        else:
            remaining -= step
            yield step
            
def find_folder(directory):
    for i in count():
        path = Path(directory) / str(i)
        if not path.is_dir():
            return str(path.absolute())

def run_fragmented_calculation(ion, target, number_ions, path, trim_settings, step):
    for i, num_ions in enumerate(fragment(step, number_ions)):
        print('total ions completed: {:06d}\tion: {}\tions in step: {:06d}'.format(i*step, ion.symbol, num_ions))
        trim_settings = trim_settings or {'calculation': 2}
        trim = TRIM(target, ion, number_ions=num_ions, **trim_settings)
        results = trim.run(SRIM_EXECUTABLE_DIRECTORY)
        save_directory = find_folder(path)
        print('calculation saved to:', save_directory)
        os.makedirs(save_directory, exist_ok=True)
        TRIM.copy_output_files(SRIM_EXECUTABLE_DIRECTORY, save_directory)

In [ ]:
# Run SRIM calcluations for each ion
for ion in ions:
    symbol_path = os.path.join(DATA_DIRECTORY, ion['identifier'])
    run_fragmented_calculation(Ion(**ion), target, NUM_IONS, symbol_path, TRIM_SETTINGS, step=IONS_PER_STEP)

total ions completed: 000000	ion: Au	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\10
total ions completed: 000020	ion: Au	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\11
total ions completed: 000040	ion: Au	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\12
total ions completed: 000060	ion: Au	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\13
total ions completed: 000080	ion: Au	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\14
total ions completed: 000000	ion: C	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\C\10
total ions completed: 000020	ion: C	ions in step: 000020
calculation saved to: C:\Users\Sacha\Desktop\nazar\examples\outputs\C\11
total ions completed: 000040	ion: C	ions in step: 000020
calculation saved to: C

In [ ]:
# Helper functions
def parse_directory(dir):
    """ Parse directory with expected structure <dir>/<ion_symbol>/<int>/COLLISON.txt
    
    This functions will collect the position <x, y, z> of every ion and recoil collision.
    
    It will write to a numpy array the parsing. This file will be about 10% of the original
    """
    if not os.path.exists(os.path.join(dir, 'collision.dat.npy')):
        positions = []
        with open(os.path.join(dir, 'COLLISON.txt'), 'rb') as f:
            for line in f.readlines():
                line = line.decode('latin-1')
                if line.endswith('Start of New Cascade  ³\r\n'):
                    tokens = line.split(chr(179))[1:-1]
                    positions.append([float(tokens[2].replace(',', '.')), float(tokens[3].replace(',', '.')), float(tokens[4].replace(',', '.'))])
                elif line.startswith('Û 0'):
                    tokens = line.split()[1:-1]
                    positions.append([float(tokens[3].replace(',', '.')), float(tokens[4].replace(',', '.')), float(tokens[5].replace(',', '.'))])
        np.save(os.path.join(dir, 'collision.dat'), np.array(positions))

def plot_heatmap(ax, data, cols, limits, norm, bins=(1000, 200)):
    """ Plot a heatmap
    
    Parameters
    ----------
    ax : maptlotlib.axes
        axis to plot heatmap on
    data : np.ndarray
        numpy array that was constructed from 'parse_directory' it is [Nx3] in dimmension.
    cols : tuple
        tuple or list length 2. example (0, 2) x, z axis
    limits : tuple
        tuple or list (2x2) [[col[0] min, col[0] max], [col[0] min, col[0] max]]
    norm : mpl.colors
        colorbar scale to use
    """
    c1, c2 = cols
    (c1min, c1max), (c2min, c2max) = limits
    region_data = data[(data[:, c1] < c1max) & (data[:, c1] > c1min) & (data[:, c2] < c2max) & (data[:, c2] > c2min)]
    heatmap, xedges, yedges = np.histogram2d(region_data[:, c1], region_data[:, c2], range=limits, bins=bins)
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    heatmap[heatmap == 0.0] = np.nan
    colormap = mpl.colormaps['viridis']
    return ax.imshow(heatmap.T, origin='lower', extent=extent, cmap=colormap, norm=norm)
    
def plot_radial_distribution_hist(ax, data, maximum, label):
    """ plot radial distribution of data
    
    Parameters
    ----------
    ax : maptlotlib.axes
        axis to plot heatmap on
    data : np.ndarray
        numpy array that was constructed from 'parse_directory' it is [Nx3] in dimmension.
    maximum : float
        maximum distance to plot radial distribution
    label : str
        label to give for plot
    """
    density, edges = np.histogram(data, bins=100, density=False, range=(1e-8, maximum))
    x = np.linspace(1e-1, maximum, len(density))
    density = [d / (2 * math.pi * r)  for r, d in zip(x, density)]
    # return ax.plot(x, density, label=label)
    return ax.semilogy(x, density, label=label)
    
def plot_radial_distribution_kde(ax, data, maximum, label):
    """ Same as plot radial distribution of data except uses kde for smoothing"""
    x = np.linspace(0, maximum, 100)
    kde = gaussian_kde(np.concatenate((-data, data)))
    return ax.plot(x, kde.pdf(x), label=label)
    
sort_by_mass = lambda e1: Element(e1).mass

In [ ]:
for dir in os.listdir(DATA_DIRECTORY):
    calculation_data = []
    total_ions = 0
    abs_dir_path = os.path.join(DATA_DIRECTORY, dir)
    if os.path.isdir(abs_dir_path):
        for calculation in os.listdir(abs_dir_path):
            if set(calculation) < set('0123456789') and os.path.isdir(os.path.join(abs_dir_path, calculation)): # check that dir only contains numbers
                parse_directory(os.path.join(abs_dir_path, calculation))
                print('parsed', os.path.join(abs_dir_path, calculation))

parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\0
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\1
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\10
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\11
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\12
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\13
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\14
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\2
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\3
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\4
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\5
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\6
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\7
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\8
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\Au\9
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\C\0
parsed C:\Users\Sacha\Desktop\nazar\examples\outputs\C\1
parsed C:\U

In [ ]:
data = {}
for dir in os.listdir(DATA_DIRECTORY):
    collisions = []
    total_ions = 0
    if os.path.isdir(os.path.join(DATA_DIRECTORY, dir)):
        for calculation in os.listdir(os.path.join(DATA_DIRECTORY, dir)):
            with open(os.path.join(DATA_DIRECTORY, dir, calculation, 'TRIM.IN')) as f:
                num_ions = int(f.read().split('\n')[2].split()[-3])
            total_ions += num_ions
            collisions.append(np.load(os.path.join(DATA_DIRECTORY, dir, calculation, 'collision.dat.npy')))
        data[dir] = {'total_ions': total_ions, 'collisions': np.concatenate(collisions) / 10}

In [ ]:
# print summary information
for ion in sorted(data, key=sort_by_mass):
    data_str = (
        "Symbol: {:2}\tNum Ions: {:7d}\tCollisions: {:7d}\n"
        "|\tMedian (x, y, z) [nm]: [{:.3f}\t{:.3f}\t{:.3f}]\n"
        "|\tMean   (x, y, z) [nm]: [{:.3f}\t{:.3f}\t{:.3f}]\n"
    )
    print(data_str.format(ion, data[ion]['total_ions'], len(data[ion]['collisions']), 
                          *np.median(data[ion]['collisions'], axis=0),
                          *np.mean(data[ion]['collisions'], axis=0)))

Symbol: C 	Num Ions:     300	Collisions:   14240
|	Median (x, y, z) [nm]: [567.200	0.279	-0.512]
|	Mean   (x, y, z) [nm]: [551.621	1.227	-1.814]

Symbol: O 	Num Ions:     300	Collisions:   24965
|	Median (x, y, z) [nm]: [533.900	0.039	-0.035]
|	Mean   (x, y, z) [nm]: [524.522	-1.220	-0.199]

Symbol: Si	Num Ions:     300	Collisions:   37277
|	Median (x, y, z) [nm]: [512.000	0.040	-0.079]
|	Mean   (x, y, z) [nm]: [527.607	-0.171	-16.306]

Symbol: Ni	Num Ions:     300	Collisions:  225573
|	Median (x, y, z) [nm]: [556.700	-0.099	-0.153]
|	Mean   (x, y, z) [nm]: [541.496	3.016	8.697]

Symbol: Au	Num Ions:     300	Collisions: 2037815
|	Median (x, y, z) [nm]: [535.100	-0.174	0.055]
|	Mean   (x, y, z) [nm]: [526.414	-0.498	-1.021]

